# Sensitivity Analysis: 2D Grid (route × sequence)

**Grid resolutions:**
- **0.05 step**: route 0.1~0.4 (7단계), sequence 0.0~0.6 (13단계) → 최대 91개 시나리오
- **0.01 step**: route 0.1~0.4 (31단계), sequence 0.0~0.6 (61단계) → 최대 1,891개 시나리오

나머지(mode) = 1 - route - sequence

출력: heatmap (ρ², FPR) + 논문 테이블 (1D sweep 포함)

In [ ]:
# Cell 1: Imports & Constants
import time
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.optimize import minimize
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings('ignore', category=FutureWarning)

ROOT = Path('../..').resolve()
DATA_DIR = ROOT / 'data' / 'training_set'
OUT_DIR = ROOT / 'data' / 'sensitivity'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Baseline
BASELINE_THRESHOLD = 0.5

# Features
TIME_RAW = ['in_vehicle_time', 'wait_time', 'access_time', 'egress_time', 'transfer_walk_time']
DIST_RAW = ['total_distance']

MODEL_FEATURES = [
    'in_vehicle_time_min', 'wait_time_min',
    'access_time_min', 'egress_time_min', 'transfer_walk_time_min',
    'total_distance_km', 'num_transfers', 'fare',
    'has_bus', 'has_train', 'has_gtx',
]
FEATURE_LABELS = [
    'IVT (min)', 'Wait (min)',
    'Access walk (min)', 'Egress walk (min)', 'Transfer walk (min)',
    'Total dist (km)', 'Transfers', 'Fare (KRW)',
    'Has bus', 'Has train', 'Has GTX',
]
SIGN_CONSTRAINED = {
    'in_vehicle_time_min', 'wait_time_min',
    'access_time_min', 'egress_time_min', 'transfer_walk_time_min',
    'total_distance_km', 'num_transfers', 'fare',
}
KEY_BETAS = ['in_vehicle_time_min', 'access_time_min', 'num_transfers', 'has_bus', 'has_train']
KEY_LABELS = ['beta_IVT', 'beta_access', 'beta_transfers', 'beta_bus', 'beta_train']

print('Setup done.')

In [2]:
# Cell 2: MNL functions

def prepare_flat_data(data, features):
    X_list, y_list, w_list, gid_list = [], [], [], []
    group_id = 0
    for _, grp in data.groupby('od_pair'):
        y = grp['choice_prob'].values.astype(np.float64)
        if abs(y.sum() - 1.0) > 0.01:
            continue
        X = grp[features].values.astype(np.float64)
        w = float(grp['n_total'].iloc[0])
        n = len(grp)
        X_list.append(X)
        y_list.append(y)
        w_list.append(np.full(n, w))
        gid_list.append(np.full(n, group_id, dtype=np.int64))
        group_id += 1
    return {
        'X': np.vstack(X_list), 'y': np.concatenate(y_list),
        'w': np.concatenate(w_list), 'gid': np.concatenate(gid_list),
        'n_groups': group_id,
    }

def mnl_neg_ll(beta, flat):
    X, y, w, gid, ng = flat['X'], flat['y'], flat['w'], flat['gid'], flat['n_groups']
    V = X @ beta
    V_max = np.full(ng, -np.inf)
    np.maximum.at(V_max, gid, V)
    V_shifted = V - V_max[gid]
    exp_V = np.exp(V_shifted)
    sum_exp = np.bincount(gid, weights=exp_V, minlength=ng)
    log_prob = V_shifted - np.log(sum_exp[gid])
    return -np.sum(w * y * log_prob)

def mnl_gradient(beta, flat):
    X, y, w, gid, ng = flat['X'], flat['y'], flat['w'], flat['gid'], flat['n_groups']
    V = X @ beta
    V_max = np.full(ng, -np.inf)
    np.maximum.at(V_max, gid, V)
    V_shifted = V - V_max[gid]
    exp_V = np.exp(V_shifted)
    sum_exp = np.bincount(gid, weights=exp_V, minlength=ng)
    prob = exp_V / sum_exp[gid]
    return X.T @ (w * (prob - y))

def compute_ll0(flat):
    gid, w, ng = flat['gid'], flat['w'], flat['n_groups']
    gs = np.bincount(gid, minlength=ng)
    wpg = np.bincount(gid, weights=w, minlength=ng) / gs
    return -np.sum(wpg * np.log(gs))

def compute_fpr(beta, flat):
    X, y, gid, ng = flat['X'], flat['y'], flat['gid'], flat['n_groups']
    V = X @ beta
    V_max = np.full(ng, -np.inf)
    np.maximum.at(V_max, gid, V)
    exp_V = np.exp(V - V_max[gid])
    sum_exp = np.bincount(gid, weights=exp_V, minlength=ng)
    pred = exp_V / sum_exp[gid]
    correct = 0
    for g in range(ng):
        m = gid == g
        if np.argmax(pred[m]) == np.argmax(y[m]):
            correct += 1
    return correct / ng

def estimate_mnl(train_flat, test_flat):
    bounds = [(None, 0) if f in SIGN_CONSTRAINED else (None, None) for f in MODEL_FEATURES]
    res = minimize(mnl_neg_ll, np.zeros(len(MODEL_FEATURES)), args=(train_flat,),
                   jac=mnl_gradient, method='L-BFGS-B', bounds=bounds,
                   options={'maxiter': 2000, 'ftol': 1e-12})
    b = res.x
    ll_b = -res.fun
    ll_0 = compute_ll0(train_flat)
    rho = 1 - ll_b / ll_0
    t_ll = -mnl_neg_ll(b, test_flat)
    t_ll0 = compute_ll0(test_flat)
    t_rho = 1 - t_ll / t_ll0
    fpr = compute_fpr(b, test_flat)
    # RMSE
    X_t, gid_t, ng_t = test_flat['X'], test_flat['gid'], test_flat['n_groups']
    V = X_t @ b
    V_max = np.full(ng_t, -np.inf)
    np.maximum.at(V_max, gid_t, V)
    exp_V = np.exp(V - V_max[gid_t])
    sum_exp = np.bincount(gid_t, weights=exp_V, minlength=ng_t)
    pred = exp_V / sum_exp[gid_t]
    rmse = np.sqrt(np.mean((pred - test_flat['y'])**2))
    return {
        'converged': bool(res.success), 'train_rho_sq': float(rho),
        'test_rho_sq': float(t_rho), 'test_fpr': float(fpr),
        'test_rmse': float(rmse),
        'betas': {f: float(v) for f, v in zip(MODEL_FEATURES, b)},
    }

print('MNL functions ready.')

MNL functions ready.


In [ ]:
# Cell 3: Load data
df = pd.read_parquet(DATA_DIR / 'route_choice_training.parquet')
print(f'{len(df):,} rows, {df["od_pair"].nunique():,} ODs')

for col in TIME_RAW:
    df[col + '_min'] = df[col] / 60
for col in DIST_RAW:
    df[col + '_km'] = df[col] / 1000

print('Data ready.')

In [ ]:
# Cell 4: Run 2D grid — 0.05 step
# route: 0.1~0.4 (7), sequence: 0.0~0.6 (13), mode = 1 - route - sequence

ROUTE_VALUES_05 = np.round(np.arange(0.10, 0.45, 0.05), 2).tolist()
SEQUENCE_VALUES_05 = np.round(np.arange(0.0, 0.65, 0.05), 2).tolist()

def recompute_composite(df, weights):
    return (weights['mode'] * df['sim_mode']
            + weights['route'] * df['sim_route']
            + weights['seq'] * df['sim_sequence'])

def filter_by_threshold(df, composite, threshold):
    df = df.copy()
    df['new_composite'] = composite
    idx_dom = df.groupby('od_pair')['choice_prob'].idxmax()
    dom_comp = df.loc[idx_dom, ['od_pair', 'new_composite']].set_index('od_pair')['new_composite']
    valid = dom_comp[dom_comp >= threshold].index
    filt = df[df['od_pair'].isin(valid)]
    od_cnt = filt.groupby('od_pair').size()
    valid2 = od_cnt[od_cnt >= 2].index
    return filt[filt['od_pair'].isin(valid2)]

def run_2d_grid(df, route_values, sequence_values, step_label):
    results = []
    t_total = time.time()
    total = sum(1 for rt in route_values for sq in sequence_values if (1.0 - rt - sq) >= -1e-9)
    done = 0

    for rt in route_values:
        for sq in sequence_values:
            w_mode = 1.0 - rt - sq
            if w_mode < -1e-9:
                continue
            w_mode = max(w_mode, 0.0)
            weights = {'mode': w_mode, 'route': rt, 'seq': sq}
            label = f'rt{rt:.2f}_sq{sq:.2f}'

            t0 = time.time()
            comp = recompute_composite(df, weights)
            filt = filter_by_threshold(df, comp, BASELINE_THRESHOLD)
            n_ods = filt['od_pair'].nunique()

            if n_ods < 100:
                done += 1
                print(f'  [{done}/{total}] [{label}] SKIP (n_ods={n_ods})')
                continue

            splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
            tr_idx, te_idx = next(splitter.split(filt, groups=filt['od_pair']))
            train_flat = prepare_flat_data(filt.iloc[tr_idx], MODEL_FEATURES)
            test_flat = prepare_flat_data(filt.iloc[te_idx], MODEL_FEATURES)

            r = estimate_mnl(train_flat, test_flat)
            r.update({'scenario': label, 'route': rt, 'sequence': sq, 'mode_weight': w_mode,
                      'n_ods': n_ods, 'weights': weights})
            for feat, key in zip(KEY_BETAS, KEY_LABELS):
                r[key] = r['betas'].get(feat, 0.0)
            results.append(r)

            done += 1
            elapsed = time.time() - t0
            print(f'  [{done}/{total}] [{label}] ODs={n_ods:,} | '
                  f'rho={r["train_rho_sq"]:.4f}/{r["test_rho_sq"]:.4f} | '
                  f'FPR={r["test_fpr"]:.3f} | {elapsed:.0f}s')

    print(f'\nDone ({step_label}): {len(results)} scenarios, {(time.time()-t_total)/60:.1f} min total')
    return results

print(f'0.05 grid: route {ROUTE_VALUES_05} × sequence {SEQUENCE_VALUES_05}')
print(f'Total combos (before feasibility check): {len(ROUTE_VALUES_05) * len(SEQUENCE_VALUES_05)}')
results_05 = run_2d_grid(df, ROUTE_VALUES_05, SEQUENCE_VALUES_05, 'step=0.05')

In [ ]:
# Cell 5: Save 0.05 grid results
df_05 = pd.DataFrame(results_05)
csv_cols = ['scenario', 'route', 'sequence', 'mode_weight', 'n_ods',
            'train_rho_sq', 'test_rho_sq', 'test_fpr', 'test_rmse',
            'beta_IVT', 'beta_access', 'beta_transfers', 'beta_bus', 'beta_train',
            'converged']
df_05[csv_cols].to_csv(OUT_DIR / 'sensitivity_2d_grid_step05.csv', index=False)

with open(OUT_DIR / 'sensitivity_2d_grid_step05.json', 'w', encoding='utf-8') as f:
    json.dump(results_05, f, indent=2, ensure_ascii=False, default=str)

print(f'Saved {len(results_05)} scenarios (step=0.05) to sensitivity_2d_grid_step05.csv / .json')
df_05[csv_cols].round(4)

In [ ]:
# Cell 5b: Run 2D grid — 0.01 step
# route: 0.1~0.4 (31), sequence: 0.0~0.6 (61), mode = 1 - route - sequence

ROUTE_VALUES_01 = np.round(np.arange(0.10, 0.41, 0.01), 2).tolist()
SEQUENCE_VALUES_01 = np.round(np.arange(0.0, 0.61, 0.01), 2).tolist()

print(f'0.01 grid: route {len(ROUTE_VALUES_01)} steps × sequence {len(SEQUENCE_VALUES_01)} steps')
total_01 = sum(1 for rt in ROUTE_VALUES_01 for sq in SEQUENCE_VALUES_01 if (1.0 - rt - sq) >= -1e-9)
print(f'Total combos (before feasibility check): {total_01}')
print(f'Estimated time: ~{total_01 * 3 / 60:.0f} hours (at ~3min/scenario)')
results_01 = run_2d_grid(df, ROUTE_VALUES_01, SEQUENCE_VALUES_01, 'step=0.01')

In [ ]:
# Cell 5c: Save 0.01 grid results
df_01 = pd.DataFrame(results_01)
csv_cols = ['scenario', 'route', 'sequence', 'mode_weight', 'n_ods',
            'train_rho_sq', 'test_rho_sq', 'test_fpr', 'test_rmse',
            'beta_IVT', 'beta_access', 'beta_transfers', 'beta_bus', 'beta_train',
            'converged']
df_01[csv_cols].to_csv(OUT_DIR / 'sensitivity_2d_grid_step01.csv', index=False)

with open(OUT_DIR / 'sensitivity_2d_grid_step01.json', 'w', encoding='utf-8') as f:
    json.dump(results_01, f, indent=2, ensure_ascii=False, default=str)

print(f'Saved {len(results_01)} scenarios (step=0.01) to sensitivity_2d_grid_step01.csv / .json')
df_01[csv_cols].round(4).head(20)

In [ ]:
# Cell 6: Heatmaps (0.05 grid)

def make_heatmap(df_2d, value_col, title, fmt, cmap, vmin=None, vmax=None):
    pivot = df_2d.pivot(index='route', columns='sequence', values=value_col)
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(pivot.values, cmap=cmap, aspect='auto',
                   vmin=vmin, vmax=vmax, origin='lower')
    # Annotate only if grid is small enough
    if pivot.shape[0] <= 15 and pivot.shape[1] <= 10:
        fontsize = max(6, 10 - max(pivot.shape[0], pivot.shape[1]) // 3)
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = pivot.values[i, j]
                if np.isnan(val):
                    continue
                vmin_eff = vmin or pivot.values[~np.isnan(pivot.values)].min()
                vmax_eff = vmax or pivot.values[~np.isnan(pivot.values)].max()
                rel = (val - vmin_eff) / (vmax_eff - vmin_eff + 1e-9)
                color = 'white' if rel > 0.6 else 'black'
                ax.text(j, i, fmt.format(val), ha='center', va='center',
                        fontsize=fontsize, fontweight='bold', color=color)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{v:.2f}' for v in pivot.columns], fontsize=7, rotation=45)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{v:.2f}' for v in pivot.index], fontsize=7)
    ax.set_xlabel('Sequence weight ($w_{sequence}$)', fontsize=12)
    ax.set_ylabel('Route weight ($w_{route}$)', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    cbar = fig.colorbar(im, ax=ax, shrink=0.85)
    plt.tight_layout()
    return fig

# --- 0.05 grid heatmaps ---
print('=== 0.05 Step Grid Heatmaps ===')
fig1 = make_heatmap(df_05, 'test_rho_sq',
    'Test McFadden $\\rho^2$ (step=0.05)',
    '{:.3f}', 'YlOrRd', vmin=0.43, vmax=0.50)
fig1.savefig(OUT_DIR / 'fig_heatmap_rho2_step05.png', dpi=150, bbox_inches='tight')
plt.show()

fig2 = make_heatmap(df_05, 'test_fpr',
    'First Preference Recovery (step=0.05)',
    '{:.1%}', 'YlGn', vmin=0.68, vmax=0.76)
fig2.savefig(OUT_DIR / 'fig_heatmap_fpr_step05.png', dpi=150, bbox_inches='tight')
plt.show()

fig3 = make_heatmap(df_05, 'n_ods',
    'Number of OD Pairs (step=0.05)',
    '{:.0f}', 'PuBu')
fig3.savefig(OUT_DIR / 'fig_heatmap_n_ods_step05.png', dpi=150, bbox_inches='tight')
plt.show()
print('0.05 heatmaps saved.')

In [ ]:
# Cell 6b: Heatmaps (0.01 grid — no annotation, too dense)

print('=== 0.01 Step Grid Heatmaps ===')
fig1 = make_heatmap(df_01, 'test_rho_sq',
    'Test McFadden $\\rho^2$ (step=0.01)',
    '{:.3f}', 'YlOrRd', vmin=0.43, vmax=0.50)
fig1.savefig(OUT_DIR / 'fig_heatmap_rho2_step01.png', dpi=150, bbox_inches='tight')
plt.show()

fig2 = make_heatmap(df_01, 'test_fpr',
    'First Preference Recovery (step=0.01)',
    '{:.1%}', 'YlGn', vmin=0.68, vmax=0.76)
fig2.savefig(OUT_DIR / 'fig_heatmap_fpr_step01.png', dpi=150, bbox_inches='tight')
plt.show()

fig3 = make_heatmap(df_01, 'n_ods',
    'Number of OD Pairs (step=0.01)',
    '{:.0f}', 'PuBu')
fig3.savefig(OUT_DIR / 'fig_heatmap_n_ods_step01.png', dpi=150, bbox_inches='tight')
plt.show()
print('0.01 heatmaps saved.')

In [ ]:
# Cell 7: β stability heatmaps (0.05 grid)

def plot_beta_heatmaps(df_grid, step_label, out_suffix):
    fig, axes = plt.subplots(1, 5, figsize=(24, 4.5))
    for ax, feat, lbl in zip(axes, KEY_BETAS, KEY_LABELS):
        df_grid[f'_tmp_{feat}'] = df_grid['betas'].apply(lambda d: d.get(feat, 0.0))
        pivot = df_grid.pivot(index='route', columns='sequence', values=f'_tmp_{feat}')
        im = ax.imshow(pivot.values, cmap='RdBu_r', aspect='auto', origin='lower')
        # Annotate only if small grid
        if pivot.shape[0] <= 15 and pivot.shape[1] <= 10:
            fs = max(5, 8 - max(pivot.shape[0], pivot.shape[1]) // 4)
            for i in range(pivot.shape[0]):
                for j in range(pivot.shape[1]):
                    val = pivot.values[i, j]
                    if not np.isnan(val):
                        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=fs)
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f'{v:.2f}' for v in pivot.columns], fontsize=6, rotation=45)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels([f'{v:.2f}' for v in pivot.index], fontsize=6)
        ax.set_xlabel('$w_{sequence}$', fontsize=9)
        ax.set_ylabel('$w_{route}$', fontsize=9)
        vals = pivot.values[~np.isnan(pivot.values)]
        cv = np.std(vals) / abs(np.mean(vals)) * 100 if abs(np.mean(vals)) > 1e-10 else 0
        ax.set_title(f'{lbl}\nCV={cv:.1f}%', fontsize=10, fontweight='bold')
        fig.colorbar(im, ax=ax, shrink=0.8)
        df_grid.drop(columns=[f'_tmp_{feat}'], inplace=True)
    fig.suptitle(f'$\\beta$ Coefficient Stability ({step_label})', fontsize=14, fontweight='bold', y=1.05)
    plt.tight_layout()
    fig.savefig(OUT_DIR / f'fig_heatmap_betas_{out_suffix}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_beta_heatmaps(df_05, 'step=0.05', 'step05')
print('Beta heatmaps (0.05) saved.')

plot_beta_heatmaps(df_01, 'step=0.01', 'step01')
print('Beta heatmaps (0.01) saved.')

In [ ]:
# Cell 8: Robustness summary (both grids)

def print_robustness(df_grid, label):
    rho_vals = df_grid['test_rho_sq'].values
    fpr_vals = df_grid['test_fpr'].values

    print('=' * 65)
    print(f'2D Grid Robustness Summary — {label}  ({len(df_grid)} scenarios)')
    print('=' * 65)
    print(f'rho2: [{rho_vals.min():.4f}, {rho_vals.max():.4f}]  spread={rho_vals.max()-rho_vals.min():.4f}')
    print(f'FPR:  [{fpr_vals.min()*100:.1f}%, {fpr_vals.max()*100:.1f}%]  spread={(fpr_vals.max()-fpr_vals.min())*100:.1f}%p')
    print()
    # Best scenario
    best_rho = df_grid.loc[df_grid['test_rho_sq'].idxmax()]
    best_fpr = df_grid.loc[df_grid['test_fpr'].idxmax()]
    print(f'  Best rho2: {best_rho["scenario"]}  (rho2={best_rho["test_rho_sq"]:.4f})')
    print(f'  Best FPR:  {best_fpr["scenario"]}  (FPR={best_fpr["test_fpr"]:.3f})')
    print()
    for feat, lbl in zip(KEY_BETAS, KEY_LABELS):
        vals = df_grid['betas'].apply(lambda d: d.get(feat, 0.0)).values
        m = np.mean(vals)
        cv = np.std(vals) / abs(m) * 100 if abs(m) > 1e-10 else 0
        print(f'  {lbl:<18}: mean={m:>10.6f}  CV={cv:>6.1f}%  {"ROBUST" if cv < 20 else "VARIABLE"}')
    print()
    print(f'All rho2 > 0.2:     {"PASS" if all(rho_vals > 0.2) else "FAIL"}')
    print(f'All FPR > 60%:      {"PASS" if all(fpr_vals > 0.6) else "FAIL"}')
    print(f'rho2 spread < 0.05: {"PASS" if (rho_vals.max()-rho_vals.min()) < 0.05 else "FAIL"}')
    print(f'FPR spread < 5%p:   {"PASS" if (fpr_vals.max()-fpr_vals.min()) < 0.05 else "FAIL"}')
    print()

print_robustness(df_05, 'step=0.05')
print_robustness(df_01, 'step=0.01')

In [ ]:
# Cell 9: Paper Table — 1D sweeps + 2D grid 요약

def fmt_table(df_in, index_col='scenario'):
    d = df_in.copy()
    d['n_ods'] = d['n_ods'].apply(lambda x: f'{x:,}')
    d['train_rho_sq'] = d['train_rho_sq'].apply(lambda x: f'{x:.4f}')
    d['test_rho_sq'] = d['test_rho_sq'].apply(lambda x: f'{x:.4f}')
    d['test_fpr'] = d['test_fpr'].apply(lambda x: f'{x*100:.1f}%')
    d['test_rmse'] = d['test_rmse'].apply(lambda x: f'{x:.4f}')
    for c in ['beta_IVT', 'beta_access', 'beta_transfers', 'beta_bus', 'beta_train']:
        d[c] = d[c].apply(lambda x: f'{x:.4f}')
    cols = [index_col, 'n_ods', 'train_rho_sq', 'test_rho_sq', 'test_fpr', 'test_rmse',
            'beta_IVT', 'beta_access', 'beta_transfers', 'beta_bus', 'beta_train']
    rename = {index_col: 'Scenario', 'n_ods': 'N(OD)', 'train_rho_sq': 'Train ρ²',
              'test_rho_sq': 'Test ρ²', 'test_fpr': 'FPR', 'test_rmse': 'RMSE',
              'beta_IVT': 'β_IVT', 'beta_access': 'β_access',
              'beta_transfers': 'β_transfers', 'beta_bus': 'β_bus', 'beta_train': 'β_train'}
    return d[cols].rename(columns=rename)

# 1D sweep tables
for fname, title in [('sensitivity_weights.csv', 'Table 1: Sensitivity to Similarity Weights (1D Sweep)'),
                      ('sensitivity_threshold.csv', 'Table 2: Sensitivity to Matching Threshold (1D Sweep)'),
                      ('sensitivity_normdist.csv', 'Table 3: Sensitivity to Normalization Distance (1D Sweep)')]:
    try:
        tmp = pd.read_csv(OUT_DIR / fname)
        print(title)
        print('=' * 100)
        display(fmt_table(tmp))
        print()
    except FileNotFoundError:
        print(f'{fname} not found\n')

# 2D grid summary
def print_grid_summary(df_grid, label):
    print(f'Table: 2D Grid Summary — {label} ({len(df_grid)} scenarios)')
    print('=' * 60)
    for m in ['test_rho_sq', 'test_fpr', 'test_rmse', 'n_ods']:
        vals = df_grid[m].values
        print(f'  {m:<16}: min={vals.min():.4f}  max={vals.max():.4f}  '
              f'mean={vals.mean():.4f}  std={vals.std():.4f}')
    for feat, lbl in zip(KEY_BETAS, KEY_LABELS):
        vals = df_grid['betas'].apply(lambda d: d.get(feat, 0.0)).values
        cv = np.std(vals) / abs(np.mean(vals)) * 100 if abs(np.mean(vals)) > 1e-10 else 0
        print(f'  {lbl:<16}: mean={np.mean(vals):.4f}  std={np.std(vals):.4f}  CV={cv:.1f}%')
    print()

print_grid_summary(df_05, 'step=0.05')
print_grid_summary(df_01, 'step=0.01')

In [ ]:
# Cell 10: LaTeX table export (논문 복붙용)

def to_latex_table(df_in, caption, label, index_col='scenario'):
    tbl = fmt_table(df_in, index_col)
    n_cols = len(tbl.columns)
    col_fmt = 'l' + 'r' * (n_cols - 1)
    lines = []
    lines.append(r'\begin{table}[htbp]')
    lines.append(r'\centering')
    lines.append(r'\caption{' + caption + '}')
    lines.append(r'\label{' + label + '}')
    lines.append(r'\scriptsize')
    lines.append(r'\begin{tabular}{' + col_fmt + '}')
    lines.append(r'\toprule')
    lines.append(' & '.join(tbl.columns) + r' \\')
    lines.append(r'\midrule')
    for _, row in tbl.iterrows():
        lines.append(' & '.join(str(v) for v in row.values) + r' \\')
    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)

# 1D sweep LaTeX
for fname, cap, lab in [
    ('sensitivity_weights.csv', 'Sensitivity analysis: similarity weight combinations', 'tab:sens_weights'),
    ('sensitivity_threshold.csv', 'Sensitivity analysis: matching threshold', 'tab:sens_threshold'),
    ('sensitivity_normdist.csv', 'Sensitivity analysis: spatial normalization distance', 'tab:sens_normdist'),
]:
    try:
        tmp = pd.read_csv(OUT_DIR / fname)
        print(to_latex_table(tmp, cap, lab))
        print()
    except FileNotFoundError:
        pass

# 2D grid top-10 LaTeX (by test_rho_sq)
for df_grid, step, suffix in [(df_05, '0.05', 'step05'), (df_01, '0.01', 'step01')]:
    top10 = df_grid.nlargest(10, 'test_rho_sq').copy()
    print(to_latex_table(top10,
        f'2D grid sensitivity: top 10 by test $\\rho^2$ (step={step})',
        f'tab:sens_2d_{suffix}_top10'))
    print()